# CS 315 · Activity 7 — Computer Vision: Face Detection + Verification
**App:** BaonTrack (budget tracker) + face verification  
**Stack:** Python · OpenCV (YuNet detector + SFace recognizer) · Streamlit · SQLite

**How it fits the activity:** laptop webcam → video stream → face detection on every frame (Streamlit *Live Face Detection* page) → face verification as a second login step (AIoT idea: a camera sensor + on-device ML decides who gets access).

Run the cells top to bottom. Colab can't see your laptop camera directly — the browser does, so the Streamlit app is exposed over an HTTPS tunnel and your browser supplies the webcam.

## 1 · Install

In [ ]:
!pip -q install streamlit streamlit-webrtc av huggingface_hub
import cv2
print("OpenCV", cv2.__version__)   # needs >= 4.8 for FaceDetectorYN / FaceRecognizerSF

## 2 · Upload the project
Upload `baontrack_face.zip` (app.py, db.py, face.py, ai_entry.py, schema.sql, requirements.txt).

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()
name = next(iter(up))
os.makedirs("/content/baontrack", exist_ok=True)
zipfile.ZipFile(name).extractall("/content/baontrack")
os.chdir("/content/baontrack")
import sys; sys.path.insert(0, "/content/baontrack")
print(os.listdir("."))

## 3 · (Optional) Hugging Face token for the AI quick-add
In Colab: 🔑 Secrets → add `HF_TOKEN` → enable notebook access. Skip this if you only care about the face feature.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("No HF_TOKEN — quick-add will be disabled, everything else works.")

## 4 · Download the face models + quick webcam test in the notebook
This cell grabs the two ONNX models, then lets you snap a photo and shows the detected face box (pure computer-vision demo, no Streamlit).

In [ ]:
import face
face.ensure_models()
print("models ready:", os.listdir("models"))

from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => capture.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    with open(filename, 'wb') as f:
        f.write(b64decode(data.split(',')[1]))
    return filename

img = cv2.imread(take_photo())
faces = face.detect_faces(img)
print(f"{len(faces)} face(s) detected")
face.draw_faces(img, faces)
cv2.imwrite("detected.jpg", img)
display(Image("detected.jpg"))

## 5 · Launch BaonTrack
Prints a public `https://…trycloudflare.com` link — open it **in your laptop's browser** and allow camera access. The link lasts until you stop this cell/runtime.

In [ ]:
import subprocess, re, threading, time, os, urllib.request

os.chdir("/content/baontrack")
subprocess.run(["pkill", "-f", "cloudflared"]); subprocess.run(["pkill", "-f", "streamlit"]); time.sleep(1)

if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
    os.chmod("cloudflared", 0o755)

st_proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        if urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2).status == 200:
            print("Streamlit is up"); break
    except Exception:
        time.sleep(1)
else:
    print(open("streamlit.log").read()[-2000:]); raise RuntimeError("Streamlit didn't start")

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--protocol", "http2", "--url", "http://localhost:8501"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url, connected, start = None, False, time.time()
for line in tunnel.stdout:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m: url = m.group(0)
    if "Registered tunnel connection" in line:
        connected = True; break
    if time.time() - start > 90: break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()
print("Tunnel connected. Open this on your laptop:" if (url and connected) else "Tunnel did not connect - run this cell again.", url)

## 6 · Demo script for the class
1. **Create Account** tab → type a username → tick consent → capture 3 face samples. The account is created and you are logged in — no password.  
2. **Live Face Detection** page → START → boxes follow your face frame by frame; green = *you*, red = *unknown*. Have a classmate step in.  
3. **Log Out** → **Face Login** tab → look at the camera → BaonTrack finds your account (1-to-N face search over everyone enrolled).  
4. Let a classmate try Face Login (no match) and try to register with *your* face (rejected: face already registered).

**Limits to mention:** a printed photo or phone screen can fool this (no liveness check); the cosine-similarity threshold (0.363) trades false accepts vs false rejects; Colab's tunnel and SQLite file are temporary — use Supabase/Postgres + pgvector for a real deployment. Old PIN accounts still work through the PIN Login tab.

## Stop everything
Run when finished.

In [ ]:
tunnel.terminate(); st_proc.terminate()